# Thuitanium — ARC-AGI-3 (a fork of the Tufa Labs duck harness)

**This is a Knowless Crew / Thuitanium submission, and it is a fork.** The ARC-AGI-3 **solver is
Tufa Labs' work** — it is mounted as an attached dataset and executed unmodified. What is ours is
the harness configuration in this notebook: the environment flags, the clock, and the diagnostics.

## Credit

The solver was written by the Tufa Labs team; in alphabetical order: Harold Bessis, Jeroen Cottaar,
Isaiah Pressman, Andries Smit, Michal Tesnar, and Stefano Viel.

- The notebook this one descends from: https://www.kaggle.com/code/jeroencottaar/taaf-duck-harness-kaggle
- Their writeup, which explains what the solver actually does: https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-3/discussion/717133
- Machine Learning Street Talk interview by Tim Scarfe about the duck harness: https://x.com/MLStreetTalk/status/2072326433922297975?s=20

⚠️ **The milestone-winning 1.21 described in the original notebook is Tufa Labs' result, not ours.**
No score on this page is theirs, and none of theirs is reported here.

## What we modified

**The solver is untouched.** Our changes are confined to the notebook's own configuration surface,
measured by diffing this fork against the upstream template rather than recalled:

- **cell 8** — the solver setup command: which model the run uses, and the environment flags that
  set its clock, its analyzer budget and its diagnostics.
- **cell 12** — the benchmark-customisation hook Tufa provides for exactly this purpose (*"make
  one-off changes to `bm`, `bm.games`, or `bm.solver` here before the run starts"*).
- **cells 2, 4, 6 and 14** — Tufa's own `__TAAF_*__` template placeholders filled in with this
  competition's wheelhouse path, working directory and dataset slugs. Every fork of the template
  does this; it is substitution, not modification.

Which lever a given build moves is stated in that build's own cell 8 / cell 12 comment and in the
`build_notebook.py` that produced it.

## What this notebook is

Infrastructure and diagnostics only — the solver code lives in the attached dataset. It installs the
ARC runtime from the competition wheelhouse, makes the bundled source snapshot importable, runs any
solver setup commands, loads the pickled benchmark, plays the competition games, and writes results
to `/kaggle/working`. Diagnostics are minimised during a real competition rerun
(`KAGGLE_IS_COMPETITION_RERUN`) and kept full otherwise.

**Note**: if you copy this notebook you must manually select the proper GPU (RTX Pro 6000).

## 1. Environment and submission mode

Detect whether this is a real competition rerun (which minimises diagnostics), set the
framework's environment flags, and put the CUDA libraries on the linker path.

In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# In submission, disable the periodic JSON/HTML diagnostics writes and per-frame logging.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")

## 2. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment
has no internet).

In [ ]:
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

## 3. Locate the source bundle

Find the uploaded TAAF source dataset by its marker file, and record where Kaggle mounted
every attached input so setup commands and the solver can find them.

In [ ]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["jakobbrggen/taaf-kaggle-source-anim-20260807-anim", "driessmit1/arc3-vllm-h100-wheelhouse-v3", "jakobbrggen/qwen3-8-27b-fp8-hf-snapshot"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir() -> Path:
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        return marker.parent
    raise RuntimeError("TAAF source bundle not found under /kaggle/input.")


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir()
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

## 4. Import the bundled source and run solver setup

Put the snapshotted repositories on the path (this process and any child processes), then run
the solver's setup commands — installing wheels, fetching model weights, and so on.

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# Solver setup commands (wheels, vLLM server startup, ...) run before the benchmark loads.
env = _command_env()
for command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    # duckv10: model swap only (R12 seam). NO output cap - v9 proved a 768-token ceiling
    # truncates the tool call that carries the action itself.
    command = (
        command
        .replace("MODEL_OWNER = 'driessmit1'", "MODEL_OWNER = 'jakobbrggen'")
        .replace(
            "MODEL_SLUG = 'vrfai-qwen3-6-27b-fp8-hf-snapshot'",
            "MODEL_SLUG = 'qwen3-8-27b-fp8-hf-snapshot'",
        )
        .replace(
            "SERVED_MODEL_NAME = 'vrfai/Qwen3.6-27B-FP8'",
            "SERVED_MODEL_NAME = 'vrfai/Qwen3.8-27B-FP8'",
        )
    )
    assert "vrfai-qwen3-6-27b-fp8-hf-snapshot" not in command, "duckv10: model slug rewrite missed"
    assert "Qwen3.6-27B-FP8" not in command, "duckv10: served-name rewrite missed"
    assert "'LOCAL_ANALYZER_MAX_OUTPUT': '0'" in command, "duckv10: output must stay UNCAPPED"
    print(f"taaf.kaggle: setup command: {command}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    # Re-read in case the command persisted new env keys.
    env = _command_env()
    os.environ.update(env)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)

## 5. Load the benchmark

Unpickle the deployment target and the benchmark, stamping the real submission state onto the
target and pointing the benchmark's outputs at the Kaggle working directory.

In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

## 6. Customization hook

Optional: tweak `bm`, `bm.games`, or `bm.solver` here before the run starts — the safe place
for one-off experiments once the deployed bundle has loaded.

In [ ]:
# duckv10: stock anim bundle (ships its own noop_guard + animation awareness);
# duckmod's patches are dropped - they target the June-era tree and measured
# zero adoption (results/wayfinder/R8).
print("duckv10: anim bundle + Qwen3.8, output UNCAPPED")


## 7. Run the benchmark

In a real competition rerun (`KAGGLE_IS_COMPETITION_RERUN`), wait for the Kaggle gateway and
play the **live competition Arcade**. Otherwise — an interactive "Save & Run" — play the
competition's **bundled environment files offline**, with no gateway required, so the notebook
runs end-to-end without a submission. Teardown commands run afterward even if the run raises.

In [ ]:
# Build the live competition game list from the gateway's available environments.
def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# Build the offline game list from the competition's bundled environment files.
def _offline_games(env_dir: str):
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    arcade = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError(f"No offline environments found under {env_dir}.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# The gateway can take a while to come up; poll until it answers.
def _wait_for_gateway(base_url: str, timeout_s: float = 600.0) -> None:
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")


# Print the run preamble and persist the launcher's git status for diagnostics.
print((BUNDLE_DIR / "preamble.txt").read_text())
(WORKING_DIR / "git_status.txt").write_text((BUNDLE_DIR / "git_status.txt").read_text())

# arc_agi reads RECORDINGS_DIR and ARC_API_KEY from env (ArcadeSpec carries neither); operation
# mode, environments dir, and base url are all passed explicitly via the spec, so no env is needed.
os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

# solo probe (B41/B42) -- NEVER-SUBMIT guard. Injected at MODULE level in cell 14, immediately
# before `if TRUE_SUBMISSION:`.
#
# WHY IT IS NOT IN solo_patch.py. The filter must live in cell 14's `else` branch, because that is
# the only place bm.games survives being reassigned. A guard spliced there is spliced into the
# branch that runs when TRUE_SUBMISSION is FALSE -- i.e. it is unreachable in exactly the case it
# names. Measured on the first build of this probe, and still true on master until this file
# existed: with the if/else intact and only the two game-list builders faked, TRUE_SUBMISSION=True
# took the gateway branch, skipped the whole solo block, and left bm.games holding all 110 live
# games with no exception raised. The run would have looked entirely normal and burned the day's
# submission on a diagnostic build.
#
# prove_teeth.py could not see this: it dedents the injected block out of its branch before
# exec'ing it, so the block's own TRUE_SUBMISSION mutation passed while the notebook skipped it.
#
# The guard therefore goes ABOVE the branch, where both paths must pass through it.
assert not TRUE_SUBMISSION, (
    "solo probe: TRUE_SUBMISSION is set. This build plays ONE game and is diagnostic; "
    "it must never be submitted. Rebuild from duckv10 if a real submission is intended."
)

if TRUE_SUBMISSION:
    # Real submission: play the live competition Arcade served by the Kaggle gateway.
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    # The gateway boots asynchronously; wait before swapping in its game list.
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    bm.games = _competition_games()
else:
    # Interactive run: play the bundled competition environments offline (no gateway).
    # The competition's environment files ship alongside the wheelhouse in the competition dataset.
    competition_env_files = str(Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels").parent / "environment_files")
    bm.games = _offline_games(competition_env_files)

    # solo probe (B41/B42) -- ONE game gets the whole clock. DIAGNOSTIC ONLY, NEVER SUBMIT.
    #
    # WHERE THIS GOES, and why not cell 12. The documented customization hook is cell 11/12, and
    # B41's ticket said the change was "one line in cell 12". That is WRONG and would have been a
    # silent no-op: cell 14 REPLACES bm.games wholesale on both paths -- the TRUE_SUBMISSION branch
    # rebuilds it from the gateway, the else branch rebuilds it from the offline environment files
    # (the two assignments are the lines just above this block; they are NOT quoted here verbatim,
    # because the builder anchors its splice on that exact text and a second copy of it inside a
    # comment makes the builder's own placement assert measure the comment instead of the code --
    # caught by solo/prove_teeth.py's placement mutation, 2026-08-26)
    # -- so anything cell 12 does to bm.games is discarded before the run. Same class as duckv21's
    # lesson that tool_agent binds build_chat_payload by name. The filter is therefore injected into
    # cell 14, anchored on the offline assignment, and runs between that line and bm.run().
    #
    # WHAT IT PROVES -- and the three figures this comment has now carried, in order, so nobody
    # copies the dead one forward again.
    #
    #   1,099 actions (B41, predicted). From "6.8 s of generation x 25 = 170 s of wall". That identity
    #   cannot support it: aggregate = N x per-request is the DEFINITION of aggregate throughput, so
    #   it reproduces the observed wall in a world where vLLM batches perfectly and solo gains nothing.
    #
    #   136-238 actions (PR #52, measured throughput). Aggregate vLLM generation throughput at one
    #   running request is 40.4 tok/s against 341.3 at twenty-five -- 40.4 vs 13.7 per request, a 3.0x
    #   speedup, not 25x. Right about the direction; still an over-prediction, because it converted a
    #   throughput ratio into an action count.
    #
    #   80 and 14 actions (B42, RUN). sk48 took 80 against a human level-1 count of 61 (1.31x) and
    #   lp85 took 14 against 17 (0.82x). Both land inside that game's own nine-run shared spread, so
    #   dropping concurrency 25 -> 1 moved the per-game action count nowhere outside historical noise.
    #   B45 names part of the gap: 24.1% (sk48) and 50.8% (lp85) of generated tokens were still in
    #   flight when the clock expired, and a throughput measurement counts those while an action
    #   counter does not.
    #
    # B42/B43 both read NO and the NO is not trusted -- the probe never delivered the budget the
    # answer rule assumed. This block is kept buildable for a re-run at a budget that does.
    _SOLO_TARGET = "g50t"

    # The never-submit guard is NOT here -- it is in solo/solo_guard.py, injected at module level
    # above `if TRUE_SUBMISSION:`. Everything in THIS file runs inside the else branch, so a guard
    # placed here is unreachable whenever TRUE_SUBMISSION is true, which is the only case it is for.
    # The comment that used to sit here also had the mechanism backwards: under TRUE_SUBMISSION this
    # filter does not "run against 110 live games", it does not run at all.

    # FILTER ON env_name, NOT game_id. `taaf.game.Game` declares
    #     game_id: str = field(default="", init=False)          (game.py:446)
    # and only `_start_game()` populates it (asserted at game.py:473), so at this point every
    # game_id is the EMPTY STRING and a prefix test on it matches nothing. Cell 14 builds these as
    # GameAPI(env_name=<id>), so env_name is the field carrying the target here. Learned the
    # expensive way: the game_id version matched 0 of 25 and fired this assert 483 s in
    # (sahasawatt/taaf-solo-sk48 v2, 2026-08-26). The rig could not have caught it -- taaf.game_api
    # needs `arcengine`, which is not installed off-Kaggle, so the fake carried whatever shape the
    # author believed. The assert now PRINTS what it saw, so the next mismatch diagnoses itself.
    _solo_before = list(bm.games)
    _solo_ids = [getattr(_g, "env_name", None) or getattr(_g, "game_id", "") for _g in _solo_before]
    bm.games = [_g for _g, _i in zip(_solo_before, _solo_ids) if str(_i).startswith(_SOLO_TARGET)]
    assert len(bm.games) == 1, (
        f"solo: expected exactly 1 game matching {_SOLO_TARGET!r}, got {len(bm.games)} "
        f"from {len(_solo_before)}; ids seen = {sorted({str(_i) for _i in _solo_ids})}"
    )
    # game_weights must stay parallel to games (benchmark.py:101). The notebook nulls it two lines
    # below, but assert rather than rely on that: a newer bundle could stop doing it.
    assert not getattr(bm, "game_weights", None), (
        "solo: bm.game_weights is set; filtering games alone would desync its length"
    )
    print(
        f"solo: {_SOLO_TARGET} only -- {len(_solo_before)} games -> 1 ({bm.games[0].game_id}); "
        f"the whole per-game clock now belongs to this game",
        flush=True,
    )


bm.n_passes = 1
bm.game_weights = None

# Outside a real submission, stop ~10 min before the wall-clock budget for a graceful exit.
soft_end = None
if not TRUE_SUBMISSION:
    budget = float(getattr(target, "max_runtime_s", 0.0) or 0.0)
    if budget > 0:
        soft_end = datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(seconds=budget - min(600.0, budget / 2))

# Play the benchmark; teardown commands run even if the run raises.
try:
    await bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=TRUE_SUBMISSION)
    if not TRUE_SUBMISSION:
        # An offline run isn't scored, but Kaggle still expects a submission.parquet output.
        import pandas as pd

        pd.DataFrame(
            [["1_0", "1", True, 1]],
            columns=["row_id", "game_id", "end_of_game", "score"],
        ).to_parquet(WORKING_DIR / "submission.parquet", index=False)
finally:
    for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
        print(f"taaf.kaggle: teardown command: {command}", flush=True)
        subprocess.run(command, shell=True, check=False, cwd=WORKING_DIR, env=_command_env())

## 8. Show the diagnostics

A non-submission run writes `diagnostics.html` to `/kaggle/working`; it is rendered inline below
(and downloadable from the working directory). You should be able to click around through the links.

In [ ]:
from html import escape

from IPython.display import HTML, display

diagnostics_html = WORKING_DIR / "diagnostics.html"
if diagnostics_html.is_file():
    # Isolate the full document in an iframe so its styles don't leak into the notebook.
    display(
        HTML(
            f'<iframe srcdoc="{escape(diagnostics_html.read_text(), quote=True)}" '
            'width="100%" height="900" style="border:0"></iframe>'
        )
    )
else:
    print("No diagnostics.html — minimal diagnostics (real submission) suppresses it.")